In [1]:
!pip install -q chromadb
!pip install -U -q "google-genai"

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


## RAG con LangChain
Ejecutamos el cuaderno con las funciones y variables necesarias para añadir datos y consultar la base de datos de Chroma utilizando la interfaz vector store de LangChain. 

In [2]:
%run ./RAG_LangChain.ipynb

---
<h2>Creacion de Agentes con Google Agent Development Kit</h2>


In [3]:
# Instalar librerías necesarias
!pip install google-adk litellm -q

import os
import requests
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# API Key y configuración
#os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

print("Configuración ADK completada.")



"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


Configuración ADK completada.


In [4]:
def obtener_peliculas_por_titulo(titulo: str):
    return query_col(query=titulo, collection='movies')[0]['metadata']

def obtener_detalles_pelicula(titulo: str):
    info = obtener_peliculas_por_titulo(titulo)
    if isinstance(info, dict) and 'movie_title' in info:
        return {
            "titulo": info.get("movie_title"),
            "director": info.get("director"),
            "cast": info.get("cast"),
            "fecha_estreno": info.get("release_date")
        }
    return info  # Devuelve Wikipedia o mensaje de error

def obtener_sinopsis_pelicula(titulo: str):
    info = obtener_peliculas_por_titulo(titulo)
    if isinstance(info, dict) and 'overview' in info:
        return info['overview']
    # Si no hay sinopsis, mirar Wikipedia
    wiki = search_in_wikipedia(titulo, True)
    if isinstance(wiki, dict):
        return wiki.get("summary", "Sinopsis no disponible.")
    return f"No se encontró información sobre '{titulo}'."

def obtener_detalles_actor(nombre: str):
    """
    Obtiene información detallada de un actor por su nombre.
    Primero intenta en TMDB y, si no encuentra resultados, busca en Wikipedia.
    """
    # 1 Buscar actor en TMDB
    url_search = f"https://api.themoviedb.org/3/search/person?query={nombre}&include_adult=false&language=en-US&page=1"
    response_search = requests.get(url_search, headers=TMDB_HEADERS)
    data_search = response_search.json()

    if data_search.get('results'):
        # Tomamos el primer resultado
        actor = data_search['results'][0]
        person_id = actor['id']

        # 2 Obtener detalles del actor usando la función existente
        detalles = get_person_details(person_id)
        gender = "Not set"
        if detalles['gender'] == 1:
            gender = "female"
        elif detalles['gender'] == 2:
            gender = "male"
        elif detalles['gender'] == 3:
            gender = "non binary"
        return {
            "nombre": detalles.get('name'),
            "biografia": detalles.get('biography') or "Sin biografía disponible",
            "fecha_nacimiento": detalles.get('birthday'),
            "fecha_fallecimiento": detalles.get('deathday'),
            "lugar_nacimiento": detalles.get('place_of_birth'),
            "departamento": detalles.get('known_for_department'),
            "genero": gender
        }

    else:
        #  3 Fallback a Wikipedia si no se encuentra en TMDB
        wiki = search_in_wikipedia(nombre)
        if isinstance(wiki, dict):
            return {
                "nombre": nombre,
                "biografia": wiki.get("summary", "Sin biografía disponible"),
                "url_wikipedia": wiki.get("url")
            }
        else:
            return f"No se encontró información sobre '{nombre}'."

    


<h2>Agente Cine</h2>

In [5]:
# Crear agente con ADK
agente_cine = Agent(
    name="AgenteCine",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Agente experto en películas, consulta TMDB y ChromaDB",
    instruction=(
        "Eres un asistente experto en películas. "
        "Puedes buscar películas por título en TMDB y almacenarlas en ChromaDB, "
        "o consultar películas almacenadas según una descripción o título. "
        "Cuando el usuario pida buscar películas, usa 'buscar_y_guardar_peliculas'. "
        "Cuando el usuario pida consultar películas, usa 'obtener_peliculas_por_titulo' o 'obtener_detalles_pelicula'."
        "Si el usuario pregunta algo sobre el argumento de la pelicula o sus reseñas (reviews) , usa 'obtener_sinopsis_pelicula'"
        "Si el resultado obtenido contiene informacion de wikipedia , filtra y devuelve al usuario solo la informacion solicitada , por ejemplo si pregunta por un director , devuelve el director de esa pelicula ."
        "Con la función query_puedes preguntarle a tu base de datos sobre la información que tienes"
    ),
    tools=[query_col, add_movies_to_collection, obtener_peliculas_por_titulo, obtener_detalles_pelicula, obtener_sinopsis_pelicula]
)

print(f"Agente '{agente_cine.name}' creado con modelo '{MODEL_GEMINI_2_0_FLASH}'.")


Agente 'AgenteCine' creado con modelo 'gemini-2.0-flash'.


<h2>Agente Actores</h2>

In [6]:
# Crear agente experto en actores
agente_actores = Agent(
    name="AgenteActores",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Agente experto en actores y actrices, consulta TMDB, ChromaDB y Wikipedia",
    instruction=(
        "Eres un asistente experto en actores y actrices. "
        "Puedes buscar actores por nombre en TMDB y almacenarlos en ChromaDB, "
        "o consultar actores almacenados según un nombre o descripción. "
        "Cuando el usuario pida información sobre un actor, usa 'obtener_detalles_actor' y devuelvele toda la informacion pedida "
        "Si no pide ningun dato en concreto , devuelvele su nombre , edad , altura , año de nacimiento y participacion en al menos 3 peliculas"
        "Si el resultado obtenido contiene información de Wikipedia, filtra y devuelve solo lo solicitado, "
        "por ejemplo la fecha de nacimiento o biografía resumida."
    ),
    tools=[add_person_to_collection, obtener_peliculas_por_titulo, search_in_wikipedia]
)

print(f"Agente '{agente_actores.name}' creado con modelo '{MODEL_GEMINI_2_0_FLASH}'.")


Agente 'AgenteActores' creado con modelo 'gemini-2.0-flash'.


In [7]:
# Crear Runner y sesión
session_service = InMemorySessionService()
APP_NAME = "cine_app"
USER_ID = "Usuario"
SESSION_ID = "user"

import asyncio

# Crear sesión async
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session creada: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

#agente de cine
runner_cine = Runner(
    agent=agente_cine,
    app_name=APP_NAME,
    session_service=session_service
)
print(f"Runner creado para el agente '{runner_cine.agent.name}'.")

#agente de actores
runner_actores = Runner(
    agent=agente_actores,
    app_name=APP_NAME,
    session_service=session_service
)

print(f"Runner creado para el agente '{runner_actores.agent.name}'.")



Session creada: App='cine_app', User='Usuario', Session='user'
Runner creado para el agente 'AgenteCine'.
Runner creado para el agente 'AgenteActores'.


<h3>Funcion para llamar a cualquier agente</h3>

In [8]:
# Función genérica para llamar a cualquier agente
async def call_any_agent_async(query: str, runner: Runner):
    print(f"\n>>> User Query: {query}")
    content = types.Content(role='user', parts=[types.Part(text=query)])
    final_response_text = "El agente no produjo respuesta final."

    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            break

    print(f"<<< Agent Response: {final_response_text}")


In [9]:
#hablar con el agente
await call_any_agent_async("Busca reviews de  Ready Player One",runner=runner_cine)
await call_any_agent_async("Dime informacion sobre LeonardoDiCaprio",runner=runner_actores)
await call_any_agent_async("Dime el reparto , director y fecha de estreno  de la pelicula Ready Player One",runner=runner_cine)




>>> User Query: Busca reviews de  Ready Player One


Event from an unknown agent: AgenteCine, event id: 51937340-6cc2-4f40-a1be-632f098afde4
Event from an unknown agent: AgenteCine, event id: afdde30f-85a4-4466-89ce-fa8551d3436b
Event from an unknown agent: AgenteCine, event id: 6e5e40ce-7a07-4f01-8259-8a6fd1fe4f6a


<<< Agent Response: Aquí tienes algunas reseñas de "Ready Player One":

*   **moovies**: "Un viaje salvaje por el camino de los recuerdos... en un DeLorean". Calificación: 7/10
*   **tmdb51616167**: "La película 'Ready Player One' ofrece un viaje nostálgico con un toque fresco". Calificación: 8/10
*   **The Movie Mob**: "¡Ready Player One es el pináculo de las películas de videojuegos con un valor de repetición extremadamente alto!". Calificación: 9/10
*   **Gimly**: "Honestamente, fue mejor de lo que esperaba...". Calificación: 5/10
*   **Per Gunnar Jonsson**: "Como nerd de la informática y fanático de la ciencia ficción, esta película me vino como anillo al dedo". Calificación: 10/10


>>> User Query: Dime informacion sobre LeonardoDiCaprio
Added these people to collection ('people'): 
Leonardo DiCaprio already exists in collection


Event from an unknown agent: AgenteActores, event id: 3d1adbe3-e353-4a83-bb74-e20886f189cf
Event from an unknown agent: AgenteActores, event id: 7feb9dbf-d5da-4eda-9906-c6ec24955011
Event from an unknown agent: AgenteActores, event id: 487ac9d6-dcd2-4844-9627-f6accdc2999d


<<< Agent Response: Leonardo DiCaprio es un actor reconocido. ¿Qué te gustaría saber sobre él? Puedo darte su nombre, edad, altura, año de nacimiento y participaciones en al menos 3 películas, o si quieres, puedo buscar información más específica.


>>> User Query: Dime el reparto , director y fecha de estreno  de la pelicula Ready Player One
<<< Agent Response: El reparto de "Ready Player One" incluye a Tye Sheridan, Olivia Cooke, Ben Mendelsohn, Lena Waithe, Simon Pegg y Mark Rylance. El director es Steven Spielberg, y la fecha de estreno fue el 28 de marzo de 2018.

